# ⚔️ LevelUp AI — Backend Server

**Setup:** Runtime → Change runtime type → **T4 GPU** → Save

After running all cells you get a public URL — paste it into the frontend:
`https://venkatkondaiahpalpu-levelup-ai.hf.space?api=YOUR_NGROK_URL`

In [ ]:
# ── Cell 1: Check GPU ────────────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q fastapi uvicorn pyngrok python-dotenv
!pip install -q torch==2.4.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q transformers peft accelerate bitsandbytes sentencepiece
!pip install -q scikit-learn joblib elevenlabs openai-whisper
print('✅ Dependencies installed')

In [ ]:
# ── Cell 3: Clone project from GitHub ────────────────────────────────────────
import os

REPO_URL = 'https://github.com/veeravenkatsaikondaiahpalpu-hue/levelup-ai.git'

if not os.path.exists('/content/levelup-ai'):
    !git clone {REPO_URL} /content/levelup-ai
else:
    !cd /content/levelup-ai && git pull

os.chdir('/content/levelup-ai')
print('✅ Project cloned')

In [ ]:
# ── Cell 4: Mount Google Drive and copy model ─────────────────────────────────
# Your LoRA adapter is NOT in the GitHub repo (too large).
# Upload the folder to Google Drive first, then run this cell.
#
# How to upload:
#   1. Open Google Drive → create folder: LevelUp/models/final/levelup-qlora-cloud
#   2. Upload ALL files from your local:
#      C:\Users\veera\Desktop\Biuld URself\levelup-ai\models\final\levelup-qlora-cloud\
#   3. Run this cell

from google.colab import drive
drive.mount('/content/drive')

import shutil, os

# Path in your Google Drive (adjust if you used a different folder name)
DRIVE_MODEL_PATH = '/content/drive/MyDrive/LevelUp/models/final/levelup-qlora-cloud'
LOCAL_MODEL_PATH = '/content/levelup-ai/models/final/levelup-qlora-cloud'

os.makedirs(os.path.dirname(LOCAL_MODEL_PATH), exist_ok=True)

if os.path.exists(DRIVE_MODEL_PATH):
    shutil.copytree(DRIVE_MODEL_PATH, LOCAL_MODEL_PATH, dirs_exist_ok=True)
    files = os.listdir(LOCAL_MODEL_PATH)
    print(f'✅ Model copied! Files: {files}')
else:
    print(f'❌ Not found at: {DRIVE_MODEL_PATH}')
    print('   Upload your model folder to Google Drive first (see instructions above)')

In [ ]:
# ── Cell 5: Configure secrets ─────────────────────────────────────────────────
import os

os.environ['HF_TOKEN']           = ''    # ← paste your HuggingFace token (for LLaMA download)
os.environ['ELEVENLABS_API_KEY'] = ''    # ← optional, enables per-build AI voices
os.environ['LEVELUP_LOAD_MODEL'] = 'true'
os.environ['LEVELUP_ADAPTER']    = 'models/final/levelup-qlora-cloud'
os.environ['WHISPER_MODEL']      = 'base'

NGROK_TOKEN = ''  # ← paste your ngrok authtoken

print('✅ Secrets configured')

In [ ]:
# ── Cell 6: Start backend + expose via ngrok ──────────────────────────────────
import subprocess, time
from pyngrok import ngrok

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

# Start FastAPI server
server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'api.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print('⏳ Starting server and loading model... (~3 min)')
time.sleep(20)

# Open public tunnel
tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url

print(f'\n🚀 Backend LIVE at:')
print(f'   {PUBLIC_URL}')
print(f'\n🌐 Full app URL:')
print(f'   https://venkatkondaiahpalpu-levelup-ai.hf.space?api={PUBLIC_URL}')
print(f'\n📖 Swagger docs: {PUBLIC_URL}/docs')
print(f'\n⚠️  Keep this tab open to stay live!')

In [ ]:
# ── Cell 7: Check server logs ─────────────────────────────────────────────────
# Run this if something seems wrong
import subprocess
result = subprocess.run(['curl', '-s', 'http://localhost:8000/health'], capture_output=True, text=True)
print('Health:', result.stdout)

# Show last 30 lines of server log
out = server.stdout.read1(4096).decode('utf-8', errors='ignore')
print('\nServer log:\n', out)

In [ ]:
# ── Cell 8: Keep alive ────────────────────────────────────────────────────────
# Prevents Colab idle disconnect — run this last
import time, subprocess
print('💓 Keep-alive running (Ctrl+C to stop)...')
while True:
    time.sleep(55)
    r = subprocess.run(['curl', '-s', 'http://localhost:8000/health'], capture_output=True, text=True)
    print(f'[{time.strftime("%H:%M")}] ✓ {r.stdout.strip()[:60]}')